In [1]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser

from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [2]:
loader = DirectoryLoader('data', glob="*.txt", show_progress=True, loader_cls=TextLoader)

data = loader.load()

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<?, ?it/s]


In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = text_splitter.split_documents(data)

In [4]:
print("Number of Documents:", len(data))
print()
print("Number of Chunks:", len(chunks))

Number of Documents: 3

Number of Chunks: 181


In [5]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma(collection_name="vector_database", 
            embedding_function=embedding_model, 
            persist_directory="./chroma_db_")
db.add_documents(documents=chunks)

C:\Users\pkracham\AppData\Local\Temp\ipykernel_26036\4016781758.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['47152870-62f2-49b6-946f-e0af8e7dd755',
 'e50a77ea-639e-4dff-819f-c098bc3aaa71',
 '0a65d84f-0f2c-4968-8677-bf0cbe2f7610',
 'b8668a6f-962a-416a-8158-21ebb90f2637',
 '47233cdc-7141-499b-9097-239374c82077',
 '7e42c3db-5892-43c8-9683-8670129cda7a',
 '7b51f7d4-79f2-4559-b64b-27fab1115dfa',
 'a5c925ca-f346-40d0-9cda-a50f418639b7',
 'a9a66434-612f-4a24-955f-bb9b961f6147',
 'd6af1b83-18ae-4dff-b3a1-e6726e5c458d',
 '9fe6c2a2-5c06-4404-81b6-aa8c98b075b8',
 'e1fd5dc4-56cd-4064-85e1-a5e899a68f5d',
 'e4b7f7e2-b3e2-4f05-8b87-968c8e6b31a9',
 'a1851abd-abbb-4e5a-b8e6-33eb963ce8db',
 '946c5478-03c4-41c9-9e33-1bef256b053d',
 'fa84e143-e176-4303-9431-34ef7dc7085f',
 '35cd684b-622f-4763-b876-1167525f75bd',
 '38353a50-65e5-4561-be64-6e4125b2dbb5',
 'fd115769-6a84-4e62-984f-d94392b30a16',
 'a9036167-0ce0-445e-b1e5-e368b1d018d9',
 '53179007-f4d7-4bf0-b25b-d3962152b749',
 'f3a1e2e6-1862-411e-b5db-ff2f681adb87',
 'eb791f33-87dc-4b86-ab85-063dda55c050',
 'e7a7ee31-fa45-4523-85fa-1537c0bdd636',
 'c965b84a-eacc-

In [6]:
print(len(db.get()["ids"]))

181


In [7]:
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [22]:
PROMPT_TEMPLATE = """
Answer the question based only on the following context:
{context}
Answer the question based on the above context: {question}.
Provide a detailed answer.
Don’t justify your answers.
Don’t give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
"""

prompt_template = ChatPromptTemplate(
    messages=[
        PROMPT_TEMPLATE
    ]
)

In [23]:
f = open('keys/groq_api_key.txt')
API_KEY = f.read()

In [24]:
groq_chat_model = ChatGroq(api_key=API_KEY, 
                           model="openai/gpt-oss-20b", 
                           temperature=1)

In [13]:
parser = StrOutputParser()

In [25]:
generator_chain = prompt_template | groq_chat_model | parser

In [16]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [26]:
rag_chain = {
    "context": retriever | format_docs, 
    "question": RunnablePassthrough()
} | generator_chain

In [28]:
query = 'What is the primary characteristic of Alzheimer’s disease?'

rag_chain.invoke(query)

'The hallmark of Alzheimer’s disease is a progressive loss of memory—particularly the difficulty remembering recent events—which stems from degeneration of the hippocampus, the brain region that supports short‑term memory.'